# 第八章配套实验：投影梯度法与投资权重优化

这个 Notebook 配合“约束优化与投影梯度法”课件使用，重点不是背公式，而是通过可运行实验观察：

1. 二维球约束下，投影梯度法如何把临时点拉回可行域；
2. 普通梯度下降为什么会走出可行域；
3. 投影如何把一个非法点拉回合法集合；
4. 投影梯度法如何处理单纯形约束；
5. 步长、投影梯度映射和收敛曲线如何影响算法表现；
6. 收益偏好参数如何改变最终投资权重。

课堂使用建议：先运行公共函数，再依次运行实验 1 到实验 6。最常修改的参数是 `lam`、`alpha`、`x0`、`max_iter`、球半径 `R` 和资产数量 `n`。

In [ ]:
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PICS = BASE / "pics"
PICS.mkdir(exist_ok=True)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 220
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]

SHANSHU = "#841e34"
BLUE = "#46445D"
DEEP_GREEN = "#0A6B3A"
ORANGE = "#AC7088"
HALF_GRAY = "0.55"

summaries = []


def add_summary(experiment, method, iterations, runtime, final_loss, note=""):
    summaries.append({
        "experiment": experiment,
        "method": method,
        "iterations": iterations,
        "runtime_sec": runtime,
        "final_loss": final_loss,
        "note": note,
    })


print(f"图片会保存到: {PICS}")

## 实验 1：二维球约束热身

先看一个二维问题，方便直接画图：

$$
\min_{\|x\|_2\le R} f(x)=\frac12(x-a)^TQ(x-a),
\qquad R=1.
$$

其中 $Q\succ0$ 不是单位矩阵，所以等高线是倾斜的椭圆；无约束最优点 $a$ 在球外。投影梯度法每一步先做普通梯度步

$$
z^k=x^k-\alpha\nabla f(x^k),
$$

再投影回球约束

$$
x^{k+1}=\Pi_{\{\|x\|_2\le R\}}(z^k).
$$

如果 $z^k$ 已经在球内，则投影不改变它；如果 $z^k$ 在球外，则沿径向拉回球边界。下面的图会同时画出当前可行点 $x^k$、临时点 $z^k$，以及投影后的 $x^{k+1}$。

In [ ]:
def ball_objective(x, Q, a):
    x = np.asarray(x, dtype=float)
    a = np.asarray(a, dtype=float)
    diff = x - a
    return 0.5 * float(diff @ Q @ diff)


def ball_grad(x, Q, a):
    x = np.asarray(x, dtype=float)
    a = np.asarray(a, dtype=float)
    return Q @ (x - a)


def project_ball(z, R=1.0):
    """Project z onto the Euclidean ball {x: ||x||_2 <= R}."""
    z = np.asarray(z, dtype=float)
    norm_z = np.linalg.norm(z)
    if norm_z <= R:
        return z.copy()
    return R * z / norm_z


def run_ball_pgd(Q=None, a=(1.35, 0.95), R=1.0, x0=(-0.75, 0.25), alpha=0.18, steps=24):
    """Return feasible iterates x^k and temporary GD points z^k for a 2D ball example."""
    if Q is None:
        Q = np.array([[4.0, 1.2], [1.2, 1.0]])
    Q = np.asarray(Q, dtype=float)
    a = np.asarray(a, dtype=float)
    x = project_ball(np.asarray(x0, dtype=float), R=R)

    xs = [x.copy()]
    zs = []
    fs = [ball_objective(x, Q, a)]

    for _ in range(steps):
        z = x - alpha * ball_grad(x, Q, a)
        x_next = project_ball(z, R=R)
        zs.append(z.copy())
        xs.append(x_next.copy())
        fs.append(ball_objective(x_next, Q, a))
        x = x_next

    return np.array(xs), np.array(zs), np.array(fs), Q, a, R


def draw_ball(ax, R=1.0):
    # Use plain text in the legend to avoid mathtext compatibility issues across Matplotlib versions.
    circle = plt.Circle((0, 0), R, color=DEEP_GREEN, alpha=0.08, label=f"feasible set ||x||_2 <= {R}")
    ax.add_patch(circle)
    boundary = plt.Circle((0, 0), R, fill=False, color=DEEP_GREEN, linewidth=2)
    ax.add_patch(boundary)
    ax.axhline(0, color="0.4", linewidth=0.6)
    ax.axvline(0, color="0.4", linewidth=0.6)
    ax.set_xlim(-1.25, 1.65)
    ax.set_ylim(-1.20, 1.35)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel(r"$x_1$")
    ax.set_ylabel(r"$x_2$")


def plot_ball_contours(ax, Q, a, xlim=(-1.25, 1.65), ylim=(-1.20, 1.35), levels=22):
    grid_x = np.linspace(xlim[0], xlim[1], 350)
    grid_y = np.linspace(ylim[0], ylim[1], 350)
    X, Y = np.meshgrid(grid_x, grid_y)
    D0 = X - a[0]
    D1 = Y - a[1]
    Z = 0.5 * (Q[0, 0] * D0**2 + 2 * Q[0, 1] * D0 * D1 + Q[1, 1] * D1**2)
    ax.contour(X, Y, Z, levels=levels, colors="0.72", linewidths=0.7)


xs_ball, zs_ball, fs_ball, Q_ball, a_ball, R_ball = run_ball_pgd()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

plot_ball_contours(axes[0], Q_ball, a_ball)
draw_ball(axes[0], R=R_ball)
axes[0].scatter([a_ball[0]], [a_ball[1]], color=ORANGE, s=90, marker="*", label=r"unconstrained minimizer $a$")
axes[0].plot(xs_ball[:, 0], xs_ball[:, 1], "o-", color=SHANSHU, linewidth=1.7, markersize=3.2, label=r"projected iterates $x^k$")

for k, z in enumerate(zs_ball[:12]):
    xk = xs_ball[k]
    xnext = xs_ball[k + 1]
    axes[0].plot([xk[0], z[0]], [xk[1], z[1]], "--", color=ORANGE, linewidth=1.0, alpha=0.75)
    axes[0].plot([z[0], xnext[0]], [z[1], xnext[1]], ":", color=BLUE, linewidth=1.4, alpha=0.9)
    axes[0].scatter([z[0]], [z[1]], s=18, color=ORANGE, alpha=0.75)

axes[0].scatter([xs_ball[0, 0]], [xs_ball[0, 1]], color=BLUE, s=55, zorder=5, label=r"start $x^0$")
axes[0].scatter([xs_ball[-1, 0]], [xs_ball[-1, 1]], color=DEEP_GREEN, s=70, zorder=6, label="final feasible point")
axes[0].set_title("球约束：临时点 z 可能出界，再径向投影回来")
axes[0].legend(loc="lower left", fontsize=8)

axes[1].plot(fs_ball, "o-", color=SHANSHU, linewidth=2, markersize=3.5)
axes[1].set_xlabel("iteration k")
axes[1].set_ylabel(r"$f(x^k)$")
axes[1].set_title("投影后可行点上的目标函数")
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
out = PICS / "chapter8_pgd_2d_ball_projection_path.png"
fig.savefig(out, bbox_inches="tight")
print(f"saved to {out}")

In [ ]:
rows = []
for k in range(len(zs_ball)):
    xk = xs_ball[k]
    zk = zs_ball[k]
    xnext = xs_ball[k + 1]
    rows.append({
        "k": k,
        "x^k": np.round(xk, 4),
        "||x^k||": np.linalg.norm(xk),
        "temporary z^k": np.round(zk, 4),
        "||z^k||": np.linalg.norm(zk),
        "z^k feasible?": bool(np.linalg.norm(zk) <= R_ball + 1e-10),
        "projected x^{k+1}": np.round(xnext, 4),
        "||x^{k+1}||": np.linalg.norm(xnext),
        "x^{k+1} feasible?": bool(np.linalg.norm(xnext) <= R_ball + 1e-10),
        "f(x^{k+1})": fs_ball[k + 1],
    })

pd.DataFrame(rows).head(12)

## 模型：带预算约束的投资组合

考虑一个简化的投资权重优化问题：

$$
\min_{x\in\Delta_n} f(x)=\frac12 x^T\Sigma x-\lambda r^T x,
$$

其中

$$
\Delta_n=\left\{x\in\mathbb{R}^n:x_i\ge0,\ \sum_{i=1}^n x_i=1\right\}.
$$

解释：

- $x_i$：第 $i$ 个资产的投资比例；
- $\Sigma$：风险矩阵，$x^T\Sigma x$ 表示组合风险；
- $r$：预期收益向量；
- $\lambda$：收益偏好，越大表示越重视收益；
- $\Delta_n$：单纯形约束，表示不能做空，且所有资金刚好分配完。

这个例子与课件中的单纯形投影直接对应，适合展示普通 GD 与投影梯度法的区别。

In [ ]:
def make_portfolio_problem(n=6, seed=8):
    """Construct a small convex portfolio problem for teaching."""
    rng = np.random.default_rng(seed)
    A = rng.normal(size=(n, n))
    Sigma = A.T @ A / n
    Sigma = Sigma + 0.25 * np.eye(n)  # make the problem clearly strongly convex
    r = rng.uniform(0.03, 0.18, size=n)
    return Sigma, r


def portfolio_f(x, Sigma, r, lam=0.8):
    x = np.asarray(x, dtype=float)
    return 0.5 * float(x @ Sigma @ x) - lam * float(r @ x)


def portfolio_grad(x, Sigma, r, lam=0.8):
    x = np.asarray(x, dtype=float)
    return Sigma @ x - lam * r


def project_simplex(z):
    """
    Project z onto the probability simplex:
        {x: x_i >= 0, sum_i x_i = 1}.

    Algorithm: sort z, find the threshold tau, then x_i=max{z_i-tau, 0}.
    """
    z = np.asarray(z, dtype=float)
    if z.ndim != 1:
        raise ValueError("z must be a one-dimensional vector")
    u = np.sort(z)[::-1]
    cssv = np.cumsum(u)
    idx = np.arange(1, len(z) + 1)
    cond = u * idx > (cssv - 1.0)
    rho = np.nonzero(cond)[0][-1]
    tau = (cssv[rho] - 1.0) / (rho + 1)
    return np.maximum(z - tau, 0.0)


def is_simplex_feasible(x, tol=1e-10):
    x = np.asarray(x, dtype=float)
    return np.all(x >= -tol) and abs(np.sum(x) - 1.0) <= tol


def simplex_violation(x):
    """A simple scalar summary of infeasibility."""
    x = np.asarray(x, dtype=float)
    negative_part = np.sum(np.maximum(-x, 0.0))
    budget_error = abs(np.sum(x) - 1.0)
    return negative_part + budget_error


def gradient_mapping_norm(x, Sigma, r, lam, alpha):
    x_next = project_simplex(x - alpha * portfolio_grad(x, Sigma, r, lam))
    return np.linalg.norm((x - x_next) / alpha)

In [ ]:
def run_plain_gd(Sigma, r, lam=0.8, alpha=0.2, x0=None, max_iter=100):
    """Plain GD ignores the simplex constraint after initialization."""
    n = len(r)
    x = np.ones(n) / n if x0 is None else np.asarray(x0, dtype=float)
    xs = [x.copy()]
    fs = [portfolio_f(x, Sigma, r, lam)]
    violations = [simplex_violation(x)]

    for _ in range(max_iter):
        x = x - alpha * portfolio_grad(x, Sigma, r, lam)
        xs.append(x.copy())
        fs.append(portfolio_f(x, Sigma, r, lam))
        violations.append(simplex_violation(x))

    return np.array(xs), np.array(fs), np.array(violations)


def run_projected_gd(Sigma, r, lam=0.8, alpha=0.2, x0=None, max_iter=500, tol=1e-8):
    """Projected GD keeps every iterate on the simplex."""
    n = len(r)
    x = np.ones(n) / n if x0 is None else project_simplex(x0)
    xs = [x.copy()]
    fs = [portfolio_f(x, Sigma, r, lam)]
    grad_maps = []
    violations = [simplex_violation(x)]

    for _ in range(max_iter):
        g = portfolio_grad(x, Sigma, r, lam)
        x_next = project_simplex(x - alpha * g)
        G = (x - x_next) / alpha

        xs.append(x_next.copy())
        fs.append(portfolio_f(x_next, Sigma, r, lam))
        grad_maps.append(np.linalg.norm(G))
        violations.append(simplex_violation(x_next))

        if np.linalg.norm(G) <= tol:
            break
        x = x_next

    return np.array(xs), np.array(fs), np.array(grad_maps), np.array(violations)


def plot_weights(ax, x, title, color=SHANSHU):
    ax.bar(np.arange(1, len(x) + 1), x, color=color, alpha=0.9)
    ax.axhline(0, color="0.25", linewidth=0.8)
    ax.set_xlabel("资产编号")
    ax.set_ylabel("权重")
    ax.set_title(title)
    ax.set_ylim(min(-0.15, np.min(x) - 0.05), max(0.7, np.max(x) + 0.08))


def summarize_solution(name, x, Sigma, r, lam):
    return {
        "method": name,
        "objective": portfolio_f(x, Sigma, r, lam),
        "sum_weights": np.sum(x),
        "min_weight": np.min(x),
        "violation": simplex_violation(x),
        "feasible": is_simplex_feasible(x),
    }

## 实验 2：普通 GD 为什么可能失效

先从一个合法的投资组合出发：

$$
x^0\in\Delta_n.
$$

普通梯度下降只做

$$
x^{k+1}=x^k-\alpha\nabla f(x^k),
$$

它不会自动保证 $x_i\ge0$，也不会自动保证 $\sum_i x_i=1$。因此，即使目标函数值下降，得到的也可能不是合法投资权重。

In [ ]:
Sigma, r = make_portfolio_problem(n=6, seed=8)
lam = 0.8
alpha = 0.35
x0 = np.ones(len(r)) / len(r)

plain_xs, plain_fs, plain_violations = run_plain_gd(
    Sigma, r, lam=lam, alpha=alpha, x0=x0, max_iter=25
)

check_points = [0, 1, 3, 10, 25]
rows = []
for k in check_points:
    xk = plain_xs[k]
    rows.append({
        "k": k,
        "f(x^k)": plain_fs[k],
        "sum_i x_i": np.sum(xk),
        "min_i x_i": np.min(xk),
        "violation": plain_violations[k],
        "feasible": is_simplex_feasible(xk),
    })

pd.DataFrame(rows)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

plot_weights(axes[0], plain_xs[0], "初始权重：合法", color=DEEP_GREEN)
plot_weights(axes[1], plain_xs[-1], "普通 GD 后：可能非法", color=SHANSHU)

fig.tight_layout()
out = PICS / "chapter8_pgd_plain_gd_infeasible_weights.png"
fig.savefig(out, bbox_inches="tight")
print(f"saved to {out}")

## 实验 3：单纯形投影的作用

对单纯形

$$
\Delta_n=\{x:x_i\ge0,\ \sum_i x_i=1\},
$$

投影问题是

$$
\Pi_{\Delta_n}(z)=\arg\min_{x\in\Delta_n}\frac12\|x-z\|^2.
$$

它的解具有形式

$$
x_i=\max\{z_i-\tau,0\},
$$

其中 $\tau$ 由 $\sum_i x_i=1$ 决定。课堂上可以理解为：整体平移，再把负数截成 0，最后让总和刚好为 1。

In [ ]:
z = np.array([0.60, 0.30, -0.40])
x_proj = project_simplex(z)

pd.DataFrame({
    "component": ["asset 1", "asset 2", "asset 3"],
    "z before projection": z,
    "Pi_Delta(z)": x_proj,
})

In [ ]:
print("投影后是否非负:", np.all(x_proj >= 0))
print("投影后总和:", x_proj.sum())
print("投影前 violation:", simplex_violation(z))
print("投影后 violation:", simplex_violation(x_proj))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
plot_weights(axes[0], z, "投影前：不是合法权重", color=ORANGE)
plot_weights(axes[1], x_proj, "投影后：回到单纯形", color=DEEP_GREEN)
fig.tight_layout()
out = PICS / "chapter8_pgd_simplex_projection_demo.png"
fig.savefig(out, bbox_inches="tight")
print(f"saved to {out}")

## 实验 4：投影梯度法求解投资权重

投影梯度法每一步做两件事：

$$
z^k=x^k-\alpha\nabla f(x^k),
$$

然后

$$
x^{k+1}=\Pi_{\Delta_n}(z^k).
$$

与普通 GD 不同，PGD 的每一个迭代点都在单纯形上，因此始终是一个合法投资组合。

In [ ]:
alpha = 0.35
start = perf_counter()
pgd_xs, pgd_fs, pgd_grad_maps, pgd_violations = run_projected_gd(
    Sigma, r, lam=lam, alpha=alpha, x0=x0, max_iter=500, tol=1e-10
)
pgd_runtime = perf_counter() - start

x_pgd = pgd_xs[-1]
add_summary("investment simplex", "PGD", len(pgd_xs) - 1, pgd_runtime, pgd_fs[-1], "simplex projection")

solution_table = pd.DataFrame([
    summarize_solution("initial", pgd_xs[0], Sigma, r, lam),
    summarize_solution("plain GD final", plain_xs[-1], Sigma, r, lam),
    summarize_solution("PGD final", x_pgd, Sigma, r, lam),
])
solution_table

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

axes[0].plot(plain_fs, color=ORANGE, linewidth=2, label="Plain GD")
axes[0].plot(pgd_fs, color=SHANSHU, linewidth=2, label="PGD")
axes[0].set_xlabel("iteration k")
axes[0].set_ylabel(r"$f(x^k)$")
axes[0].set_title("目标函数曲线")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].semilogy(np.maximum(plain_violations, 1e-16), color=ORANGE, linewidth=2, label="Plain GD")
axes[1].semilogy(np.maximum(pgd_violations, 1e-16), color=SHANSHU, linewidth=2, label="PGD")
axes[1].set_xlabel("iteration k")
axes[1].set_ylabel("simplex violation")
axes[1].set_title("可行性违反程度")
axes[1].grid(True, which="both", alpha=0.3)
axes[1].legend()

axes[2].semilogy(pgd_grad_maps, color=DEEP_GREEN, linewidth=2)
axes[2].set_xlabel("iteration k")
axes[2].set_ylabel(r"$\|G_\alpha(x^k)\|$")
axes[2].set_title("投影梯度映射")
axes[2].grid(True, which="both", alpha=0.3)

fig.tight_layout()
out = PICS / "chapter8_pgd_convergence_and_feasibility.png"
fig.savefig(out, bbox_inches="tight")
print(f"PGD iterations = {len(pgd_xs)-1}, runtime = {pgd_runtime:.4f}s")
print(f"saved to {out}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
plot_weights(axes[0], plain_xs[-1], "普通 GD 最终点：不一定合法", color=ORANGE)
plot_weights(axes[1], x_pgd, "PGD 最终点：合法投资组合", color=DEEP_GREEN)
fig.tight_layout()
out = PICS / "chapter8_pgd_final_weights_comparison.png"
fig.savefig(out, bbox_inches="tight")
print(f"saved to {out}")

## 实验 5：步长对投影梯度法的影响

步长 $\alpha$ 仍然很重要：

- 太小：每一步移动很少，收敛慢；
- 合适：目标函数稳定下降；
- 太大：可能震荡，甚至目标函数不稳定。

这里对比几个不同步长下的 PGD 表现。

In [ ]:
alphas = [0.03, 0.12, 0.35, 0.80]
step_results = {}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for a in alphas:
    start = perf_counter()
    xs_a, fs_a, gm_a, viol_a = run_projected_gd(
        Sigma, r, lam=lam, alpha=a, x0=x0, max_iter=300, tol=1e-8
    )
    runtime = perf_counter() - start
    step_results[a] = (xs_a, fs_a, gm_a, viol_a)
    add_summary("step size comparison", f"PGD alpha={a}", len(xs_a) - 1, runtime, fs_a[-1], "different alpha")

    axes[0].plot(fs_a, linewidth=2, label=fr"$\alpha={a}$")
    axes[1].semilogy(np.maximum(gm_a, 1e-16), linewidth=2, label=fr"$\alpha={a}$")

axes[0].set_xlabel("iteration k")
axes[0].set_ylabel(r"$f(x^k)$")
axes[0].set_title("不同步长下的目标函数")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_xlabel("iteration k")
axes[1].set_ylabel(r"$\|G_\alpha(x^k)\|$")
axes[1].set_title("不同步长下的投影梯度映射")
axes[1].grid(True, which="both", alpha=0.3)
axes[1].legend()

fig.tight_layout()
out = PICS / "chapter8_pgd_stepsize_comparison.png"
fig.savefig(out, bbox_inches="tight")
print(f"saved to {out}")

## 实验 6：收益偏好 $\lambda$ 如何改变权重

参数 $\lambda$ 控制收益项的重要程度：

$$
f(x)=\frac12x^T\Sigma x-\lambda r^Tx.
$$

当 $\lambda$ 较小时，算法更重视风险分散；当 $\lambda$ 较大时，算法更愿意把权重分配给高收益资产。

In [ ]:
lams = [0.0, 0.4, 0.8, 1.6]
fig, axes = plt.subplots(1, len(lams), figsize=(14, 3.6), sharey=True)

lambda_rows = []
for ax, lam_value in zip(axes, lams):
    xs_lam, fs_lam, gm_lam, viol_lam = run_projected_gd(
        Sigma, r, lam=lam_value, alpha=0.35, x0=x0, max_iter=500, tol=1e-10
    )
    x_final = xs_lam[-1]
    plot_weights(ax, x_final, fr"$\lambda={lam_value}$", color=SHANSHU)
    lambda_rows.append({
        "lambda": lam_value,
        "objective": fs_lam[-1],
        "iterations": len(xs_lam) - 1,
        "largest_weight": np.max(x_final),
        "asset_with_largest_weight": int(np.argmax(x_final) + 1),
        "feasible": is_simplex_feasible(x_final),
    })

fig.suptitle("收益偏好改变最终投资权重", y=1.02)
fig.tight_layout()
out = PICS / "chapter8_pgd_lambda_weights.png"
fig.savefig(out, bbox_inches="tight")
print(f"saved to {out}")
pd.DataFrame(lambda_rows)

## 实验总结

通过这个 Notebook 可以看到：

1. 普通 GD 不会自动尊重约束，可能产生负权重或预算不等于 1；
2. 单纯形投影把不可行点拉回最近的合法投资组合；
3. 投影梯度法每一步都保持可行，适合“目标光滑、约束投影容易”的问题；
4. 投影梯度映射 $\|G_\alpha(x^k)\|$ 可以作为停止准则；
5. 步长和收益偏好参数会明显影响算法轨迹和最终解。

课堂提问建议：

- 为什么普通 GD 的目标函数可能下降，但解却不能使用？
- 单纯形投影和“简单截断负数”有什么区别？
- 如果约束不是单纯形，而是复杂非线性约束，投影梯度法还方便吗？

In [ ]:
pd.DataFrame(summaries)